In [1]:
import torch
import numpy as np
from tqdm import tqdm
from collections import defaultdict 
from poincare import PoincareManifold          # tes fichiers locaux
from model import Distance_PE

In [2]:
# Modifier à chaque fois :
checkpoint = torch.load('models/poincare_hpo_0.5_50.pt', map_location='cpu', weights_only=False)

objects = checkpoint['objects']
node2id = checkpoint['node2id']
losses = checkpoint['losses']
norm_history = checkpoint['norm_history']
edges = checkpoint['edges']
data = checkpoint['data']
hp = checkpoint['hyperparams']

In [3]:
manifold = PoincareManifold()
model = Distance_PE(n=len(objects), dim=hp['dim'],
                       manifold=manifold, sparse=True)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

Distance_PE(
  (embeddings): Embedding(19389, 2, sparse=True)
)

In [4]:
W = model.weight.detach().cpu().numpy()   # (N, dim)
norms = np.linalg.norm(W, axis=1)

print(f"Modèle chargé — {len(objects)} nœuds | dim={hp['dim']} | "
      f"{len(losses)} epochs")
print(f"Norme moy={norms.mean():.4f} | max={norms.max():.4f}")

i_min = np.argmin(norms).item()
print(f"Index de la plus petite norme : {i_min}")
print(f"Valeur de la norme : {W[i_min]}")


Modèle chargé — 19389 nœuds | dim=2 | 50 epochs
Norme moy=0.4489 | max=0.8413
Index de la plus petite norme : 851
Valeur de la norme : [ 0.0106935 -0.0074312]


In [6]:
degrees = np.array([len(data.pos_neighbors[i]) for i in range(len(objects))])
norms = model.weight.detach().norm(dim=-1).numpy()

# Corrélation degré/norme attendue : négative
from scipy.stats import spearmanr
rho, pval = spearmanr(degrees, norms)
print(f"Corrélation Spearman degré/norme : {rho:.3f} (p={pval:.2e})")

Corrélation Spearman degré/norme : 0.395 (p=0.00e+00)


In [7]:
pos_neighbors = defaultdict(set)
pos_parents = defaultdict(set)

for u, v in edges:
    pos_neighbors[int(u)].add(int(v))
    pos_parents[int(v)].add(int(u))
    

len(pos_parents[i_min])
len(pos_parents[0])

19388

In [8]:
degrees = np.array([len(data.pos_neighbors[i]) for i in range(len(objects))])

# Top 10 plus proches du centre
center_ids = np.argsort(norms)[:10]
print("=== 10 nœuds les plus proches du CENTRE ===")
for i in center_ids:
    print(f"  {objects[i]:<30} norme={norms[i]:.4f}  degré={degrees[i]}")

# Top 10 plus proches du bord
border_ids = np.argsort(norms)[-10:]
print("\n=== 10 nœuds les plus proches du BORD ===")
for i in border_ids:
    print(f"  {objects[i]:<30} norme={norms[i]:.4f}  degré={degrees[i]}")

=== 10 nœuds les plus proches du CENTRE ===
  HP:0007550                     norme=0.0130  degré=11
  HP:0000975                     norme=0.0133  degré=8
  HP:0430015                     norme=0.0155  degré=11
  HP:0030166                     norme=0.0159  degré=3
  HP:0007480                     norme=0.0162  degré=6
  HP:0000966                     norme=0.0168  degré=7
  HP:0025276                     norme=0.0175  degré=17
  HP:0001069                     norme=0.0178  degré=6
  HP:0000001                     norme=0.0180  degré=19388
  HP:0007451                     norme=0.0184  degré=4

=== 10 nœuds les plus proches du BORD ===
  HP:0034340                     norme=0.8311  degré=4
  HP:0032113                     norme=0.8317  degré=3
  HP:6000094                     norme=0.8318  degré=2
  HP:0003831                     norme=0.8322  degré=3
  HP:0040284                     norme=0.8325  degré=2
  HP:6001251                     norme=0.8326  degré=2
  HP:6000302              

In [10]:
from sklearn.metrics import average_precision_score

def evaluate(model, objects, edges, node2id):
    model.eval()
    W  = model.weight.detach()   # (N, dim)
 
    pos_neighbors = defaultdict(set)
    for u, v in edges:
        pos_neighbors[int(u)].add(int(v))

    ranksum, ap_scores = 0, 0
    nranks = 0
    iters = 0
    labels = np.empty(model.embeddings.weight.size(0))
 
    for u in tqdm(objects):
        labels.fill(0)
        u = int(node2id[u])
        neighbors = pos_neighbors.get(u, set())
        if not neighbors :
            continue
        u_exp = W[u].unsqueeze(0).expand(W.shape[0], -1)  # Coorconnées de u dans la boule de Poincaré
        dists = manifold.distance(u_exp, W).numpy()  # Distance de u aux autres noeuds
        dists[u] = 1e12
        #order = np.argsort(dists)  # Tri par distance décroissante p/r à u
        sorted_ind = np.argsort(dists)

        #ranks = int(np.where(order == v)[0][0]) + 1  # Rang du noeud v p/r à u dans l'embedding
        #ranks.append(rank)
        ranks, = np.where(np.isin(sorted_ind, list(neighbors)))
        ranks += 1
        N = ranks.shape[0]

        ranksum += ranks.sum() - (N * (N - 1) / 2)
        nranks += ranks.shape[0]
        labels[list(neighbors)] = 1
        ap_scores += average_precision_score(labels, -dists)
        iters += 1

        #pos  = pos_neighbors[u]  # Voisins de u dans la représentation initiale
        #hits, psum = 0, 0.0 
        #for k, idx in enumerate(order[1:], 1):  # On parcourt les noeuds du plus proche au plus éloigné
            #if idx in pos:
                #hits  += 1
                #psum  += hits / k
        #aps.append(psum / max(len(pos), 1))
 
    return float(ranksum), nranks, ap_scores, iters

In [ ]:
results = evaluate(model, objects, edges, node2id)

print("Erreur moyenne sur le rang : ", float(results[0]) / results[1])
print("MAP :", float(results[2]) / results[3])

100%|██████████| 19389/19389 [10:11<00:00, 31.70it/s] 

Erreur moyenne sur le rang :  1220.9928709153644
Erreur MAP moyenne : 0.07170192425297756
